# Project status

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JeffVallyath/geometry-of-truth/blob/v1.1.1/notebooks/00_project_status.ipynb)

## Research question

The project asks whether a language model encodes the relation between a situation and a moral consideration, and whether that internal relation predicts how the model's answer changes under meaning-preserving rephrasing. The first claim concerns relation geometry. The second concerns predictive value beyond text and the model's own answer margin.

Three pieces are complete. The ValuePrism split and checkerboard controls define an evaluation that resists recognized consideration shortcuts. A factual positive control shows that the activation method recovers truth across answer mappings. A Llama 3.1 8B Instruct development test then finds a moral-relation signal at layer 19.

Two pieces remain open. Human reviewers have not completed the semantic calibration needed for the confirmatory checkerboards, and the sealed confirmatory model result remains unopened. The rephrasing-flip experiment has not run. Current claims stop at development evidence.

## Contents

1. Stage table
2. Moral-relation development design
3. Main interaction and control results
4. Native-confidence comparison
5. Claim boundary and source lineage

## Start here

Click the Open in Colab badge, then choose Runtime and Run all. This notebook verifies and presents public aggregate artifacts on CPU in minutes. It downloads neither Llama nor licensed ValuePrism rows.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PUBLIC_REPOSITORY = 'https://github.com/JeffVallyath/geometry-of-truth.git'
PUBLIC_REF = 'v1.1.1'
PUBLIC_COMMIT = 'cf605a169eef6cbe24ead242e0a5a39097df4f0d'
RUN_MODE = 'DEMO'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = Path("/content/geometry-of-truth")
    if (REPO_ROOT / ".git").is_dir():
        origin = subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"], check=True, text=True, capture_output=True).stdout.strip()
        if origin.rstrip("/").removesuffix(".git") != PUBLIC_REPOSITORY.rstrip("/").removesuffix(".git"):
            raise RuntimeError("The existing checkout has an unexpected origin")
        dirty = subprocess.run(["git", "-C", str(REPO_ROOT), "status", "--porcelain"], check=True, text=True, capture_output=True).stdout
        if dirty:
            raise RuntimeError("The existing checkout contains modified or untracked files")
        subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", PUBLIC_COMMIT], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", PUBLIC_COMMIT], check=True)
    elif not (REPO_ROOT / "pyproject.toml").is_file():
        if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
            raise RuntimeError("The Colab repository directory exists but is not a usable checkout")
        REPO_ROOT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "init"], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "add", "origin", PUBLIC_REPOSITORY], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", PUBLIC_COMMIT], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
    head = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip()
    if head != PUBLIC_COMMIT:
        raise RuntimeError(f"Expected public commit {PUBLIC_COMMIT}, found {head}")
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(path for path in candidates if (path / "pyproject.toml").is_file())

if RUN_MODE == "ANALYSIS":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-truth-analysis.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_ROOT), "--no-deps"], check=True)
elif RUN_MODE == "FULL":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-truth-reproduction.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_ROOT), "--no-deps"], check=True)
elif IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_ROOT)], check=True)

OUTPUT_ROOT = Path(os.environ.get("GEOMETRY_OUTPUT_ROOT", "/content/geometry-results" if IN_COLAB else str(REPO_ROOT / "geometry-results")))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ROOT = str(REPO_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

print({"mode": RUN_MODE, "output_root": str(OUTPUT_ROOT)})

In [ ]:
from IPython.display import display

from geometry_of_truth.project.contracts import load_status
from geometry_of_truth.project.results import (
    development_design,
    development_intervals,
    development_results,
    source_lineage,
    stage_table,
)

bundle = load_status(REPO_ROOT)
status = bundle["status"]
print({"artifact_integrity": "verified", "as_of": status["as_of"]})

## Full experimental arc

Each stage protects a different inference. The split limits memorization across training and testing. Checkerboards cancel fixed additive situation and consideration preferences. The factual control checks the extraction and probing procedure against a known semantic distinction. The moral development slice tests the target relation. Human audit and rephrasing then determine whether the result survives semantic review and predicts behavior under new wording.

In [ ]:
display(stage_table(status))

## Moral-relation development design

The model sees a situation and consideration together. A difference-in-means direction and a logistic activation probe score whether the consideration Supports or Opposes the action. The primary board statistic I_b sums the two within-situation differences on a reciprocal checkerboard. Fixed additive situation-only or consideration-only scores give I_b=0.

The frozen development slice uses 1,500 training rows, 300 selection rows across 75 boards, and 500 evaluation rows across 125 boards. Development selection chooses layer 19. The evaluation rows determine the measurements below, so this is a completed development result rather than the human-audited confirmation.

In [ ]:
display(development_design(status))
display(development_results(status))

## Interaction result and controls

The difference-in-means direction gives I_b=1.65, while the logistic activation probe gives I_b=2.08. The frozen SBERT matched-text baseline gives I_b=0.28. Situation-only and consideration-only controls give I_b=0, and the separate-encoding additive control is numerically zero.

For both activation methods, none of 200 group-preserving permutation values match the observed improvement over SBERT. The add-one value is p=1/201. The 95 percent interval for the difference-in-means advantage over SBERT runs from 1.09 to 1.64. The logistic advantage runs from 1.50 to 2.10.

In [ ]:
display(development_intervals(status))

## Native-confidence comparison

AUROC measures relation decoding on individual development rows. Native answer margin reaches 0.721. The difference-in-means activation direction reaches 0.732, and the logistic activation probe reaches 0.780. The simple direction only slightly exceeds native confidence, while the learned activation probe has a larger gap.

These AUROCs do not measure rephrasing-flip prediction. That experiment will ask whether the original hidden state predicts later answer changes after controlling for text and native confidence. Treating the current 0.780 result as a flip-prediction result would cross the project claim boundary.

## Provenance and next decision point

The tables come from a hash-verified public status artifact. The source hashes below identify the retained development result and its independent audit archive without exposing local paths or row text.

In [ ]:
display(source_lineage(status))
print(status["claim_boundary"])

The next material decision follows the blind human calibration. Its result determines whether the checkerboard pool can support the confirmatory endpoint and whether the sealed relation test can be opened under the frozen protocol. Rephrasing remains a separate experiment even if confirmation succeeds.